In [1]:
import os
%pwd

'c:\\Users\\singh\\OneDrive\\Desktop\\IMDB Projetc\\research'

In [ ]:
# os.chdir('../')
%pwd

'c:\\Users\\singh\\OneDrive\\Desktop\\IMDB Projetc'

In [3]:
import re
import pickle
import numpy as np
import logging

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

from src.IMDBSentimentAnalysis.entity.config_entity import DataTransformationConfig, ModelTrainerConfig, Params

logger = logging.getLogger(__name__)


class PredictionPipeline:
    def __init__(
        self,
        transform_config : DataTransformationConfig,
        trainer_config   : ModelTrainerConfig,
        params           : Params
    ):
        self.transform_config = transform_config
        self.trainer_config   = trainer_config
        self.params           = params
        self.model            = None
        self.tokenizer        = None


    # ── Step 1: Load Model ────────────────────────────────────────
    def load_model(self):
        try:
            self.model = load_model(self.trainer_config.model_path)
            logger.info(f"✅ Model loaded      : {self.trainer_config.model_path}")

        except Exception as e:
            logger.error(f"❌ Model load failed : {e}")
            raise e


    # ── Step 2: Load Tokenizer ────────────────────────────────────
    def load_tokenizer(self):
        try:
            with open(self.transform_config.tokenizer_path, 'rb') as f:
                self.tokenizer = pickle.load(f)
            logger.info(f"✅ Tokenizer loaded  : {self.transform_config.tokenizer_path}")

        except Exception as e:
            logger.error(f"❌ Tokenizer load failed: {e}")
            raise e


    # ── Step 3: Clean Text ────────────────────────────────────────
    def clean_text(self, text: str) -> str:
        text = re.sub(r'<.*?>', '', text)           # remove HTML tags
        text = re.sub(r'[^a-zA-Z\s]', '', text)    # remove special chars
        text = text.lower().strip()
        return text


    # ── Step 4: Preprocess Input ──────────────────────────────────
    def preprocess(self, review: str) -> np.ndarray:
        try:
            cleaned  = self.clean_text(review)
            sequence = self.tokenizer.texts_to_sequences([cleaned])
            padded   = pad_sequences(
                sequence,
                maxlen    = self.params.max_len,
                padding   = 'post',
                truncating= 'post'
            )
            return padded

        except Exception as e:
            logger.error(f"❌ Preprocessing failed: {e}")
            raise e


    # ── Step 5: Predict Single Review ────────────────────────────
    def predict(self, review: str) -> dict:
        try:
            if self.model is None:
                self.load_model()
            if self.tokenizer is None:
                self.load_tokenizer()

            padded      = self.preprocess(review)
            probability = self.model.predict(padded, verbose=0)[0][0]
            label       = "Positive 😊" if probability >= 0.5 else "Negative 😞"
            confidence  = probability if probability >= 0.5 else 1 - probability

            result = {
                "review"     : review,
                "sentiment"  : label,
                "confidence" : round(float(confidence) * 100, 2),
                "probability": round(float(probability), 4)
            }

            logger.info(f"✅ Sentiment  : {result['sentiment']}")
            logger.info(f"   Confidence : {result['confidence']}%")

            return result

        except Exception as e:
            logger.error(f"❌ Prediction failed: {e}")
            raise e


    # ── Step 6: Predict Batch Reviews ────────────────────────────
    def predict_batch(self, reviews: list) -> list:
        try:
            if self.model is None:
                self.load_model()
            if self.tokenizer is None:
                self.load_tokenizer()

            results = []
            for review in reviews:
                result = self.predict(review)
                results.append(result)

            logger.info(f"✅ Batch predicted   : {len(results)} reviews")
            return results

        except Exception as e:
            logger.error(f"❌ Batch prediction failed: {e}")
            raise e

In [4]:
import logging
from src.IMDBSentimentAnalysis.config.configuration import ConfigurationManager
# from src.IMDBSentimentAnalysis.components.prediction import PredictionPipeline

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

STAGE_NAME = "Prediction Stage"


class PredictionStagePipeline:
    def __init__(self):
        config                = ConfigurationManager()
        self.transform_cfg    = config.get_data_transformation_config()
        self.trainer_cfg      = config.get_model_trainer_config()
        self.params           = config.get_params()

        self.pipeline = PredictionPipeline(
            transform_config = self.transform_cfg,
            trainer_config   = self.trainer_cfg,
            params           = self.params
        )

        # Preload model and tokenizer once
        self.pipeline.load_model()
        self.pipeline.load_tokenizer()


    def predict_single(self, review: str) -> dict:
        return self.pipeline.predict(review)


    def predict_batch(self, reviews: list) -> list:
        return self.pipeline.predict_batch(reviews)


# ── Test it directly ──────────────────────────────────────────────
if __name__ == "__main__":
    try:
        logger.info(f">>>>>> {STAGE_NAME} started <<<<<<")

        predictor = PredictionStagePipeline()

        # ── Single Review ─────────────────────────────────────────
        review = "This movie was absolutely fantastic! The acting was brilliant."
        result = predictor.predict_single(review)

        print("\n" + "="*55)
        print(f"  Review     : {result['review'][:60]}...")
        print(f"  Sentiment  : {result['sentiment']}")
        print(f"  Confidence : {result['confidence']}%")
        print(f"  Probability: {result['probability']}")
        print("="*55)

        # ── Batch Reviews ─────────────────────────────────────────
        batch_reviews = [
            "The plot was terrible and the acting was unbearable.",
            "One of the best films I have ever seen in my life!",
            "It was okay, nothing special, just average movie.",
            "Absolutely loved every minute of this masterpiece!",
            "Worst movie ever. Complete waste of time and money."
        ]

        print("\n── Batch Predictions ──────────────────────────────────")
        results = predictor.predict_batch(batch_reviews)
        for i, res in enumerate(results, 1):
            print(f"  {i}. {res['sentiment']}  ({res['confidence']}%)  |  {res['review'][:50]}...")
        print("="*55)

        logger.info(f">>>>>> {STAGE_NAME} completed <<<<<<\n\nx==========x")

    except Exception as e:
        logger.exception(e)
        raise e

[ 2026-05-18 23:40:37,043 : INFO : __init__ : Logger initialized successfully ]
[ 2026-05-18 23:40:37,063 : INFO : 3615453982 : >>>>>> Prediction Stage started <<<<<< ]
[ 2026-05-18 23:40:37,065 : INFO : main : YAML file: config\config.yaml Loaded Successfully ]
[ 2026-05-18 23:40:37,068 : INFO : main : YAML file: params.yaml Loaded Successfully ]
[ 2026-05-18 23:40:37,069 : INFO : main : Create Directory at: artifacts ]
[ 2026-05-18 23:40:37,070 : INFO : main : Create Directory at: artifacts/data_transformation ]
[ 2026-05-18 23:40:37,071 : INFO : main : Create Directory at: artifacts/model_trainer ]
[ 2026-05-18 23:40:37,236 : WARNING : config : TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin. ]
[ 2026-05-18 23:40:37,246 : WARNING : saving_utils : Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` wi